# 37. Tokens, subword tokenization, and context windows

**Level:** Intermediate

**Prerequisites:** 32 and 34: text processing and embeddings.

[Course index](00_START_HERE.ipynb) | [Previous](36_Transformer_Blocks_and_Positions.ipynb) | [Next](38_Language_Model_Pretraining_and_Next_Tokens.ipynb)

**Topics:** LLM, Tokens, Tokenization, Subword tokenization, BPE, Context window, Token budget.

**Matching script:** `examples\37_Tokens_Subwords_and_Context_Windows.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A token is a model vocabulary unit, not necessarily a word or a fixed number of characters.
Tokenizers can encode spaces, punctuation, bytes, or subword pieces. The same text can have
different token counts under different vocabularies and normalization rules.

Byte-pair encoding begins with smaller units and learns frequent adjacent merges. Real
implementations specify normalization, byte mapping, merge order, special tokens, and decoding.
The toy merger below demonstrates one merge, not a compatible tokenizer for any hosted model.

The context window limits the tokenized input/output state a model supports under its service
and architecture. Reserve room for generated output and any service-managed reasoning or tool
overhead according to the actual model contract. A larger window does not guarantee attention
to every detail, perfect recall, or unlimited output length.

Check exact token counts with the selected model's tokenizer. Token-based price and memory
planning cannot be accurately derived from word count alone.

## Mathematics or data contract

A simplified request budget is input_tokens + reserved_output_tokens <= context_limit; real APIs may add special overhead.

## Important limitation

The character-based count and one-step BPE are explicit toys, not production token accounting.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: One BPE merge and a checked context budget

**Purpose:** Make token units and overflow explicit.

### Read the code in this order
1. Count adjacent symbol pairs.
2. Merge the most frequent pair in a deterministic order.
3. Verify the merged symbols still reconstruct the text.
4. Reject an over-budget request rather than silently truncating.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
overflow rejected
learned toy merge: ('l', 'o') tokens: ['lo', 'w', 'e', 'r']
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
from collections import Counter

corpus = [list("lower"), list("lowest"), list("lower")]
counts = Counter(pair for symbols in corpus for pair in zip(symbols, symbols[1:]))
chosen = sorted(counts, key=lambda pair: (-counts[pair], pair))[0]
def merge(symbols, pair):
    output, index = [], 0
    while index < len(symbols):
        if index + 1 < len(symbols) and tuple(symbols[index:index+2]) == pair:
            output.append("".join(pair))
            index += 2
        else:
            output.append(symbols[index])
            index += 1
    return output
merged = [merge(symbols, chosen) for symbols in corpus]
assert ["".join(symbols) for symbols in merged] == ["lower", "lowest", "lower"]
def check_budget(input_tokens, reserved_output, limit):
    if min(input_tokens, reserved_output) < 0 or input_tokens + reserved_output > limit:
        raise ValueError("request exceeds toy context budget")
    return limit - input_tokens - reserved_output
assert check_budget(6, 3, 10) == 1
try:
    check_budget(9, 3, 10)
except ValueError:
    print("overflow rejected")
else:
    raise AssertionError("overflow accepted")
print("learned toy merge:", chosen, "tokens:", merged[0])

overflow rejected
learned toy merge: ('l', 'o') tokens: ['lo', 'w', 'e', 'r']


### Interpret example 1

The result should demonstrate this purpose: **Make token units and overflow explicit.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Train several merge steps on a tiny corpus and preserve round-trip decoding for every example.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `LLM` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A simplified request budget is input_tokens + reserved_output_tokens <= context_limit; real APIs may add special overhead.
- Name the boundary of the demonstration: The character-based count and one-step BPE are explicit toys, not production token accounting.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/1706.03762`
- `https://huggingface.co/docs/tokenizers/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.